**LOADING THE DATACUBE AND DATACUBE VIEWER**

In [12]:
from odc.ui import DcViewer
from datacube import Datacube

dc = Datacube(env='default')

#you can create different environments inside datacube to avoid package conflicts.
#you can modify the time, center location, the height and width, and the style.Keep in mind the date format and location format (yyyy-mm-dd, (lat, lon) or (N,S))

dcv = DcViewer(
    dc, 
    time='2023-04-24',
    zoom=10,
    center=(11.05, 75.65),
    height='500px', width='1000px',
    products='non-empty',
    style={'fillOpacity': 0.05,
            'color': 'teal',
            'weight': 0.7})
dcv

In [13]:
from IPython.display import display
import ipyleaflet as L
from ipywidgets import widgets as w
import numpy as np

from datacube import Datacube
from datacube.utils.rio import set_default_rio_config

from odc.ui import show_datasets, mk_image_overlay, with_ui_cbk

In [14]:
RGB=('red', 'green','blue')

set_default_rio_config(aws={'region_name': 'auto'}, 
                       cloud_defaults=True)

qq = dict(lat=(11.05, 11.35),
          lon=(75.65, 75.95),
          time='2023-08-14')
dss = dc.find_datasets(product='sentinel_2_c1_l2a', 
                       **qq)
#here 'sentinel_2_c1_l2a' is the product name I have given to my product you should change if you get a product not found error.

**SENTINEL SURFACE REFLENCTANCE PRODUCT LOADER**

In [4]:

xx = dc.load(product='sentinel_2_l2a',
        datasets=dss,
        output_crs='EPSG:32643',  # This is what Leaflet uses by default
        measurements=RGB,
        group_by='solar_day',
        resolution=(-200, 200),
        progress_cbk=with_ui_cbk())
xx

#keep the resolution to optimum where it doesn't take much time.

<xarray.Dataset> Size: 2MB
Dimensions:      (time: 1, y: 550, x: 550)
Coordinates:
  * time         (time) datetime64[ns] 8B 2023-08-14T05:35:55.960000
  * y            (y) float64 4kB 1.3e+06 1.3e+06 1.3e+06 ... 1.19e+06 1.19e+06
  * x            (x) float64 4kB 4.999e+05 5.001e+05 ... 6.095e+05 6.097e+05
    spatial_ref  int32 4B 32643
Data variables:
    red          (time, y, x) uint16 605kB 0 0 0 0 0 ... 3123 3201 3511 4656
    green        (time, y, x) uint16 605kB 0 0 0 0 0 ... 3245 3302 3562 4665
    blue         (time, y, x) uint16 605kB 0 0 0 0 0 ... 3248 3269 3506 4601
Attributes:
    crs:           EPSG:32643
    grid_mapping:  spatial_ref

In [5]:
m = show_datasets(dss,
                  style={'opacity': 0.3, 'fillOpacity': 0},
                  width='1800px', 
                  height='800px', 
                  scroll_wheel_zoom=True)

In [6]:
img_layer = mk_image_overlay(xx, clamp=3000, fmt='png')
m.add_layer(img_layer)

In [7]:
slider = w.FloatSlider(min=0, max=1, value=1,        # remember opacity is valid in [0,1] range
                       orientation='vertical',       # Vertical slider is what I am using here
                       readout=False,                # no need to show exact value
                       layout=w.Layout(width='2em')) # Fine tune display layout, make it thinner

# Connect slider value to opacity property of the Image Layer
w.jslink((slider, 'value'),         
         (img_layer, 'opacity') )
m.add_control(L.WidgetControl(widget=slider))

In [8]:
display(m)

Map(center=[11.262724343398451, 75.50366962506848], controls=(ZoomControl(options=['position', 'zoom_in_text',…